# Plate detection — A1 (YOLO26n vs YOLOv8n) — Kaggle
Thin driver (OS-independent). All logic lives in `plate_detect`; this notebook only calls the CLI.

**Kaggle runtime only.** Setup clones the **public** repo, installs the CLI, and points raw A1 at the attached Kaggle Dataset. All later cells run from the repo root at `/kaggle/working`.

**Before running:**
1. Notebook settings → **Internet: ON** (git clone + pip; requires phone-verified account).
2. Notebook settings → **Accelerator: GPU** (T4×2 or P100).
3. **Add data** → search `duydieunguyen/licenseplates` (auto-attached via kernel-metadata `dataset_sources`).
4. Repo `UIT-DoAnCuoiKi/UIT2026-DoAnCuoiKi` must be **public** (clone uses no token). If it's private, add a `GH_PAT` secret instead — but Kaggle Secrets are NOT carried by `kaggle kernels push`; they only work from the web editor.

In [ ]:
# === Setup (Kaggle) ===
# Clones the PUBLIC repo BRANCH into /kaggle/working, installs CLI, symlinks raw A1 to the attached dataset.
import os, glob, shutil, socket

assert os.path.isdir("/kaggle"), "this notebook is Kaggle-only"

# --- fail early, with a clear reason, before any slow work ---
# 1) internet
try:
    socket.create_connection(("github.com", 443), timeout=5).close()
except OSError:
    raise SystemExit(
        "Internet is OFF on this kernel. Settings -> Internet: On "
        "(requires phone verification: kaggle.com/settings). Enable, then re-run."
    )
# 2) GPU present AND its arch is supported by Kaggle's torch. Kaggle's torch 2.10+cu128 dropped
#    Pascal (sm_60) -> a P100 gives 'CUDA no kernel image'. Require a T4 (sm_75) or newer.
import torch
print("torch", torch.__version__)
assert torch.cuda.is_available(), "CUDA not available — Settings -> Accelerator: GPU"
cap  = torch.cuda.get_device_capability()
name = torch.cuda.get_device_name(0)
archs = torch.cuda.get_arch_list()
print(f"GPU: {name}  sm_{cap[0]}{cap[1]}  | torch archs: {archs}")
if f"sm_{cap[0]}{cap[1]}" not in archs:
    raise SystemExit(
        f"GPU {name} (sm_{cap[0]}{cap[1]}) NOT supported by torch {torch.__version__} (built for {archs}). "
        "Settings -> Accelerator -> 'GPU T4 x2' (NOT P100), or push with --accelerator NvidiaTeslaT4. Then re-run."
    )
torch.zeros(1, device="cuda") + 1   # proof a kernel actually launches
print("GPU kernel launch OK")

# 3) dataset attached — auto-locate the data root (images/train) anywhere under /kaggle/input
hits = glob.glob("/kaggle/input/**/images/train", recursive=True)
if not hits:
    have = sorted(glob.glob("/kaggle/input/*"))
    raise SystemExit(
        "A1 dataset not attached (no */images/train under /kaggle/input). "
        "Add data -> duydieunguyen/licenseplates, then re-run. "
        f"Currently attached: {have or 'nothing'}"
    )
DATA_ROOT = os.path.abspath(hits[0][: -len("/images/train")])
print("A1 data root:", DATA_ROOT)

REPO   = "/kaggle/working/UIT2026-DoAnCuoiKi"
BRANCH = "feat/plate-detect-a1"                 # package NOT merged to main yet — clone this branch
RAW    = "data/raw/kaggle_vn_plate_segment"     # layout the A1Adapter expects: {images,labels}/{train,val}
URL    = "https://github.com/UIT-DoAnCuoiKi/UIT2026-DoAnCuoiKi.git"  # public repo, no token needed

# --- clone the public repo, feature branch ---
if not os.path.isdir(REPO):
    !git clone --branch {BRANCH} --single-branch {URL} {REPO}
assert os.path.isdir(REPO), "clone failed — is the repo public? (make UIT2026-DoAnCuoiKi public, or use a PAT)"
%cd {REPO}
!git rev-parse --abbrev-ref HEAD   # confirm the feature branch is checked out

# --- install the package -> puts the `plate_detect` CLI on PATH (Kaggle's torch is kept as-is) ---
!pip install -q -e src/ml/plate_detection_pipeline

# --- symlink RAW -> the located dataset root ---
if not os.path.isdir(f"{RAW}/images/train"):
    os.makedirs(os.path.dirname(RAW), exist_ok=True)
    if os.path.islink(RAW) or os.path.exists(RAW):
        (os.unlink if os.path.islink(RAW) else shutil.rmtree)(RAW)
    os.symlink(DATA_ROOT, os.path.abspath(RAW))
    print("raw A1 ->", DATA_ROOT)

# sanity-check the raw layout the adapter reads (train + val, images + labels)
for s in ("train", "val"):
    for k in ("images", "labels"):
        assert os.path.isdir(f"{RAW}/{k}/{s}"), f"missing {RAW}/{k}/{s} — check split names (val vs valid)"
print("OK — CLI installed, raw A1 ready at", RAW)

In [ ]:
import torch; print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 1. Prepare (class-map gate → split → dedup train↔test & train↔val → validate)

In [ ]:
!plate_detect prepare

In [ ]:
!plate_detect check

## 2. Train — full matrix @640 (both models × seeds 0,1,2)

In [ ]:
# FULL RUN — default config (configs/default.yaml: imgsz 640, epochs 50, patience 10, seeds 0,1,2), both models
!plate_detect train --config src/ml/plate_detection_pipeline/configs/default.yaml --project runs

## 4. Export best → ONNX (per model & imgsz), parity-checked

In [ ]:
# example; repeat per model/imgsz best run:
!plate_detect export --weights runs/yolo26n_s0_640/weights/best.pt --out weights/yolo26n_a1_640.onnx --imgsz 640

## 5. Evaluate on A1 test → comparison table + experiments.csv

In [ ]:
!plate_detect eval --imgszs 640 --project runs --weights-dir weights --sample-image data/processed/a1_det/images/test/$(ls data/processed/a1_det/images/test | head -1)

## 6. Package results

Zips `runs/`, `weights/`, and `experiments.csv` into one archive under `/kaggle/working/`. Kaggle persists `/kaggle/working` automatically — the zip appears in the notebook **Output** tab, downloadable after the session. No Drive mount needed.

In [ ]:
# === Package all results into one .zip under /kaggle/working ===
import os, datetime

STAMP   = datetime.datetime.now().strftime("%Y%m%d_%H%M")
ARCHIVE = f"/kaggle/working/plate_det_results_{STAMP}.zip"

# collect what exists (runs/ = weights+plots+curves, weights/ = exported ONNX, experiments.csv)
targets = [p for p in ("runs", "weights", "experiments.csv") if os.path.exists(p)]
assert targets, "nothing to zip — run train/export/eval first"
print("zipping:", targets)
!zip -rq "{ARCHIVE}" {" ".join(targets)}
print("archive:", ARCHIVE, f"({os.path.getsize(ARCHIVE)/1e6:.1f} MB)")
print("grab it from the notebook Output tab after the session ends.")